In [1]:
# Le "!" permet d'exécuter des commandes système dans le notebook
!git clone https://github.com/Houcineaberhache/Real-time-object-detection-.git
%cd Real-time-object-detection-

Cloning into 'Real-time-object-detection-'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 44 (delta 10), reused 35 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 49.14 MiB | 31.21 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/Real-time-object-detection-


In [14]:
%%writefile houcine.py
# Cell 1 — Check GPU
import torch

print("Torch version :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("GPU name       :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Cell 2 — Install dependencies
# NOTE: run this once in a Colab cell before importing this file:
#   !pip install ultralytics -q
import subprocess
subprocess.run(["pip", "install", "ultralytics", "-q"], check=True)

print("✓ Ultralytics (YOLOv8) installed")

# Cell 3 — Imports
from ultralytics import YOLO
import cv2
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Image
import numpy as np

print("✓ All libraries imported")

# Cell 4 — Load pretrained YOLOv8 model
model = YOLO("yolov8n.pt")  # downloads ~6MB automatically

print("✓ Model loaded")
print(f"  Running on : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


# Cell 5 — Target classes (COCO IDs relevant to our project)

TARGET_CLASSES = {
    0  : "person",
    9  : "traffic light",
    11 : "stop sign",
    2  : "car",
    7  : "truck",
}

# Bounding box colors (RGB for matplotlib)
CLASS_COLORS = {
    0  : (255, 200, 0),    # person       → yellow
    9  : (0,   255, 100),  # traffic light → green
    11 : (255, 50,  50),   # stop sign    → red
    2  : (150, 150, 150),  # car          → gray
    7  : (100, 100, 100),  # truck        → dark gray
}

print("✓ Target classes defined:")
for id, name in TARGET_CLASSES.items():
    print(f"   Class {id:2d} → {name}")

# Cell 6 — Core detector function (your main contribution)

def detect(frame, conf_threshold=0.45):
    """
    Run YOLOv8 on a single frame.

    Args:
        frame          : BGR image (numpy array from OpenCV)
        conf_threshold : minimum confidence to keep a detection

    Returns:
        detections : list of dicts [{class_id, class_name, confidence, bbox}]
        annotated  : frame with bounding boxes drawn
    """
    results = model.predict(
        source=frame,
        conf=conf_threshold,
        device="cuda" if torch.cuda.is_available() else "cpu",
        verbose=False,
    )

    detections = []
    annotated = frame.copy()

    for result in results:
        for box in result.boxes:
            class_id = int(box.cls[0])

            # Keep only our target classes
            if class_id not in TARGET_CLASSES:
                continue

            confidence = float(box.conf[0])
            class_name = TARGET_CLASSES[class_id]
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            detections.append({
                "class_id"   : class_id,
                "class_name" : class_name,
                "confidence" : confidence,
                "bbox"       : (x1, y1, x2, y2),
            })

            # Draw bounding box
            color = CLASS_COLORS.get(class_id, (255, 255, 255))
            color_bgr = (color[2], color[1], color[0])  # convert to BGR for OpenCV
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color_bgr, 2)

            label = f"{class_name} {confidence:.2f}"
            (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
            cv2.rectangle(annotated, (x1, y1 - lh - 8), (x1 + lw + 4, y1), color_bgr, -1)
            cv2.putText(annotated, label, (x1 + 2, y1 - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 1)

    return detections, annotated


def show_frame(frame, title="Detection result"):
    """Helper to display a BGR frame nicely inside Colab."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 7))
    plt.imshow(rgb)
    plt.title(title, fontsize=13)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

print("✓ detect() function ready")

Overwriting houcine.py


In [39]:
import cv2  # Bibliothèque pour le traitement d'image et vidéo
import numpy as np  # Pour les calculs mathématiques et les matrices
from scipy.spatial import distance as dist  # Pour calculer les distances entre piétons
from collections import OrderedDict  # Pour garder l'ordre des piétons détectés
from google.colab.patches import cv2_imshow  # Outil spécial pour afficher des images sur Colab
from IPython.display import clear_output  # Pour effacer l'image précédente et créer un effet vidéo

# Import du détecteur depuis houcine.py
from houcine import detect

# ─────────────────────────────────────────────────────────────────────────────
# TRACKER
# ─────────────────────────────────────────────────────────────────────────────
class CentroidTracker:
    def __init__(self, max_disappeared=25, max_distance=150):
        self.next_id      = 0
        self.objects      = OrderedDict()   # {id: (cx,cy)}
        self.bboxes       = OrderedDict()   # {id: (x1,y1,x2,y2)}
        self.disappeared  = OrderedDict()   # {id: frames_missing}
        self.max_disappeared = max_disappeared
        self.max_distance    = max_distance  # Rejette les appariements trop lointains

    def register(self, centroid, bbox):
        self.objects[self.next_id]    = centroid
        self.bboxes[self.next_id]     = bbox
        self.disappeared[self.next_id] = 0
        self.next_id += 1

    def update(self, detections):
        # detections : liste de ((cx,cy), (x1,y1,x2,y2))

        if len(detections) == 0:
            for oid in list(self.disappeared):
                self.disappeared[oid] += 1
                if self.disappeared[oid] > self.max_disappeared:
                    del self.objects[oid]
                    del self.bboxes[oid]
                    del self.disappeared[oid]
            return self.objects, self.bboxes

        new_centroids = np.array([d[0] for d in detections], dtype="int")
        new_bboxes    = [d[1] for d in detections]

        if len(self.objects) == 0:
            for i in range(len(new_centroids)):
                self.register(new_centroids[i], new_bboxes[i])
        else:
            ids       = list(self.objects.keys())
            old_cents = list(self.objects.values())

            D    = dist.cdist(np.array(old_cents), new_centroids)
            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]

            used_rows, used_cols = set(), set()
            for row, col in zip(rows, cols):
                if row in used_rows or col in used_cols:
                    continue
                if D[row, col] > self.max_distance:
                    continue
                oid = ids[row]
                self.objects[oid]    = new_centroids[col]
                self.bboxes[oid]     = new_bboxes[col]
                self.disappeared[oid] = 0
                used_rows.add(row)
                used_cols.add(col)

            for col in set(range(len(new_centroids))) - used_cols:
                self.register(new_centroids[col], new_bboxes[col])

            for row in set(range(len(ids))) - used_rows:
                oid = ids[row]
                self.disappeared[oid] += 1
                if self.disappeared[oid] > self.max_disappeared:
                    del self.objects[oid]
                    del self.bboxes[oid]
                    del self.disappeared[oid]

        return self.objects, self.bboxes


# ─────────────────────────────────────────────────────────────────────────────
# HELPER
# ─────────────────────────────────────────────────────────────────────────────
def feet_in_polygon(bbox, polygon):
    """Test si les PIEDS (bas-centre de la bbox) sont dans le polygone."""
    x1, y1, x2, y2 = bbox
    feet_x = (x1 + x2) // 2
    feet_y = y2
    return cv2.pointPolygonTest(polygon, (float(feet_x), float(feet_y)), False) >= 0


# ─────────────────────────────────────────────────────────────────────────────
# INITIALISATION
# ─────────────────────────────────────────────────────────────────────────────
PATH_VIDEO = "/content/Real-time-object-detection-/Video.mp4"

tracker = CentroidTracker(max_disappeared=90, max_distance=250)

# ─────────────────────────────────────────────────────────────────────────────
# ZONE D'INTÉRÊT — PASSAGE PIÉTON UNIQUEMENT (calibrée sur Video.mp4 1280×682)
#
# Ce polygone couvre EXACTEMENT les bandes zébrées, trottoirs exclus :
#
#         A(420,455)──────────────B(870,455)
#        /   ← passage arrière-plan →      \
#       /                                   \
#  D(90,682)──────────────────────C(1050,682)
#      ← passage premier plan →
#
# Trottoir GAUCHE : x < 90  à y=682  → exclu (D démarre à x=90)
# Trottoir DROIT  : x > 1050 à y=682 → exclu (C s'arrête à x=1050)
# ─────────────────────────────────────────────────────────────────────────────
ROI_POLYGON = np.array([
    [ 420,  455],   # A — haut-gauche  : bord gauche passage arrière-plan
    [ 870,  455],   # B — haut-droit   : bord droit  passage arrière-plan
    [1050,  682],   # C — bas-droit    : bord droit  passage premier plan  (avant trottoir droit)
    [  90,  682],   # D — bas-gauche   : bord gauche passage premier plan  (après trottoir gauche)
], dtype=np.int32)

# ─────────────────────────────────────────────────────────────────────────────
# COMPTAGE
#
# La logique est : compter un piéton une seule fois, quand il a TRAVERSÉ.
# Un piéton « a traversé » quand :
#   1. Ses pieds entrent dans le polygone ROI (il est sur le passage piéton)
#   2. Il s'est déplacé d'au moins MIN_TRAVEL_PX pixels horizontalement
#      depuis sa première apparition dans la ROI
#      → filtre les gens qui s'arrêtent au bord ou attendent le feu
#
# Stockage :
#   roi_entry_x[id]  : position x des pieds quand l'ID est entré dans la ROI
#   counted_ids      : set des IDs déjà comptés (garantit unicité)
#   crossing_count   : total piétons comptés
# ─────────────────────────────────────────────────────────────────────────────
MIN_TRAVEL_PX  = 65    # déplacement horizontal minimum pour valider une traversée
roi_entry_x    = {}    # {id: x_at_entry}
counted_ids    = set() # IDs déjà comptés
crossing_count = 0

# ─────────────────────────────────────────────────────────────────────────────
# BOUCLE PRINCIPALE
# ─────────────────────────────────────────────────────────────────────────────
cap   = cv2.VideoCapture(PATH_VIDEO)
vid_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
vid_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps   = int(cap.get(cv2.CAP_PROP_FPS))
out   = cv2.VideoWriter('video_final.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (vid_w, vid_h))

print(f"Résolution : {vid_w}×{vid_h}  FPS : {fps}")
print("Analyse en cours...")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # ── Détection YOLO sur chaque frame, seuil bas pour ne rater personne
    detections_raw, _ = detect(frame, conf_threshold=0.30)

    # ── Filtre : garder uniquement les piétons dont les PIEDS sont dans la ROI
    #    (trottoirs exclus géométriquement par le polygone calibré)
    in_roi_detections = []
    for d in detections_raw:
        if d["class_id"] != 0:          # 'person' uniquement
            continue
        if feet_in_polygon(d["bbox"], ROI_POLYGON):
            x1, y1, x2, y2 = d["bbox"]
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            in_roi_detections.append(((cx, cy), (x1, y1, x2, y2)))

    # ── Mise à jour du tracker avec uniquement les piétons dans la ROI
    tracked_objects, tracked_bboxes = tracker.update(in_roi_detections)

    # ── Logique de comptage : entrée dans la ROI + déplacement minimal
    for obj_id, (cx, cy) in tracked_objects.items():

        x1, y1, x2, y2 = tracked_bboxes[obj_id]
        feet_x = (x1 + x2) // 2

        # Première apparition de cet ID dans la ROI → mémorise la position d'entrée
        if obj_id not in roi_entry_x:
            roi_entry_x[obj_id] = feet_x

        # Ne compter que si pas encore compté ET déplacement suffisant
        if obj_id not in counted_ids:
            travel = abs(feet_x - roi_entry_x[obj_id])
            if travel >= MIN_TRAVEL_PX:
                counted_ids.add(obj_id)
                crossing_count += 1

    # ── Dessin des piétons détectés dans la ROI
    for obj_id, (cx, cy) in tracked_objects.items():
        x1, y1, x2, y2 = tracked_bboxes[obj_id]
        already_counted = obj_id in counted_ids

        # Vert = déjà compté comme traversant | Jaune = dans la ROI, pas encore compté
        color = (0, 220, 0) if already_counted else (0, 220, 220)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"ID {obj_id}", (x1, y1 - 7),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        cv2.circle(frame, (cx, cy), 4, color, -1)

    # ── ROI (overlay rouge semi-transparent + contour)
    overlay = frame.copy()
    cv2.fillPoly(overlay, [ROI_POLYGON], (0, 0, 140))
    cv2.addWeighted(overlay, 0.22, frame, 0.78, 0, frame)
    cv2.polylines(frame, [ROI_POLYGON], isClosed=True, color=(0, 0, 255), thickness=2)

    # ── Compteur en temps réel
    cv2.rectangle(frame, (10, 10), (410, 62), (0, 0, 0), -1)
    cv2.putText(frame, f"Pedestrians crossed: {crossing_count}", (16, 47),
                cv2.FONT_HERSHEY_SIMPLEX, 1.05, (0, 255, 255), 2)

    out.write(frame)

# ─────────────────────────────────────────────────────────────────────────────
# RÉSUMÉ CONSOLE
# ─────────────────────────────────────────────────────────────────────────────
cap.release()
out.release()

print("=" * 50)
print(f"  Total traversées comptées : {crossing_count}")
print(f"  IDs ayant traversé        : {sorted(counted_ids)}")
print("=" * 50)
print("Téléchargez 'video_final.mp4' dans l'onglet fichiers.")

Résolution : 1280×682  FPS : 23
Analyse en cours...
  Total traversées comptées : 19
  IDs ayant traversé        : [4, 5, 6, 7, 8, 9, 10, 13, 14, 15, 17, 18, 22, 30, 38, 41, 46, 47, 54]
Téléchargez 'video_final.mp4' dans l'onglet fichiers.
